# 05 · sktime: one interface for many forecasters

[sktime](https://www.sktime.net) gives every forecaster the same `fit(y, X) → predict(fh)` API, plus **composable** building blocks:
- transformations,
- pipelines,
- reductions,
- ensembles,
- tuning,
- evaluation.

The value is less in any single model than in being able to build and swap them like Lego.

In [ ]:
%load_ext autoreload
%autoreload 2
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (12, 4)

In [ ]:
from tsforecasting.data.data_loader import load_airline, load_jena, temporal_split
from tsforecasting.features.build_features import TARGET

try:
    df = load_jena()
except FileNotFoundError:  # first run: download + clean + features
    from tsforecasting.data import make_dataset
    from tsforecasting.features import build_features
    make_dataset.main(); build_features.main()
    df = load_jena()
y = df[TARGET]
airline = load_airline()
df.shape, airline.shape

In [ ]:
from sktime.forecasting.base import ForecastingHorizon
from sktime.split import temporal_train_test_split
from sktime.utils.plotting import plot_series

y_air = airline.copy()
y_air.index = y_air.index.to_period("M")
y_train, y_test = temporal_train_test_split(y_air, test_size=24)
fh = ForecastingHorizon(y_test.index, is_relative=False)
fh

## 1. The forecasting horizon `fh`

`fh` can be **relative** (steps ahead, $h = 1,\dots,H$) or **absolute** (timestamps). Every forecaster accepts both.

In [ ]:
from sktime.forecasting.naive import NaiveForecaster
from sktime.forecasting.theta import ThetaForecaster
from sktime.forecasting.exp_smoothing import ExponentialSmoothing

models = {
    "naive (last season)": NaiveForecaster(strategy="last", sp=12),
    "theta": ThetaForecaster(sp=12),
    "holt-winters (mul)": ExponentialSmoothing(trend="add", seasonal="mul", sp=12),
}
preds = {k: m.fit(y_train).predict(fh) for k, m in models.items()}
plot_series(y_train, y_test, *preds.values(), labels=["train", "test", *preds], markers=[""] * (2 + len(preds)));

## 2. Reduction: any regressor becomes a forecaster

`make_reduction` builds the lag matrix (window length $p$) and wraps a scikit-learn regressor. The `strategy` is `recursive`, `direct` or `multioutput`.

Trees can't **extrapolate**: their prediction is an average of training targets, so a trending series must be detrended first. That is our motivation for pipelines.

In [ ]:
import lightgbm as lgb
from sktime.forecasting.compose import make_reduction

reg = make_reduction(lgb.LGBMRegressor(n_estimators=200, min_child_samples=5, verbose=-1),
                     window_length=24, strategy="recursive")
p_raw = reg.fit(y_train).predict(fh)
plot_series(y_train, y_test, p_raw, labels=["train", "test", "LightGBM on raw y (can't extrapolate)"],
            markers=["", "", ""]);

## 3. Pipelines: transform → forecast → inverse-transform

`TransformedTargetForecaster` chains transformers on $y$ before the forecaster and inverts them on the predictions:

$$\log \;\rightarrow\; \text{deseasonalize}(m=12) \;\rightarrow\; \text{detrend} \;\rightarrow\; \text{LightGBM}$$

Each step removes one piece of structure we diagnosed in notebook 01, leaving the regressor a stationary problem.

In [ ]:
from sktime.forecasting.compose import TransformedTargetForecaster
from sktime.forecasting.trend import PolynomialTrendForecaster
from sktime.transformations.series.boxcox import LogTransformer
from sktime.transformations.series.detrend import Deseasonalizer, Detrender

pipe = TransformedTargetForecaster([
    ("log", LogTransformer()),
    ("deseason", Deseasonalizer(sp=12)),
    ("detrend", Detrender(PolynomialTrendForecaster(degree=1))),
    ("lgbm", make_reduction(lgb.LGBMRegressor(n_estimators=200, min_child_samples=5, verbose=-1),
                            window_length=12, strategy="recursive")),
])
p_pipe = pipe.fit(y_train).predict(fh)
plot_series(y_train, y_test, p_raw, p_pipe, labels=["train", "test", "raw LightGBM", "pipeline LightGBM"],
            markers=[""] * 4);

## 4. Ensembles

Averaging forecasts from diverse models usually beats picking one: the "forecast combination puzzle" (Bates & Granger, 1969). If the errors are not perfectly correlated, the mean of $K$ forecasts has lower variance.

In [ ]:
from sktime.forecasting.compose import EnsembleForecaster

ens = EnsembleForecaster([("theta", ThetaForecaster(sp=12)),
                          ("hw", ExponentialSmoothing(trend="add", seasonal="mul", sp=12)),
                          ("pipe", pipe)])
p_ens = ens.fit(y_train).predict(fh)

## 5. Evaluate everything with `evaluate`

sktime's `evaluate` runs rolling-origin backtesting (cf. notebook 02) with a **splitter**. `ExpandingWindowSplitter` reproduces our protocol: initial window 96, $H=12$, step 6.

In [ ]:
from sktime.forecasting.model_evaluation import evaluate
from sktime.performance_metrics.forecasting import MeanAbsoluteScaledError
from sktime.split import ExpandingWindowSplitter

cv = ExpandingWindowSplitter(initial_window=96, fh=list(range(1, 13)), step_length=6)
candidates = {**models, "lgbm pipeline": pipe, "ensemble": ens}
results = {k: evaluate(m, cv, y_air, scoring=MeanAbsoluteScaledError(sp=12)) for k, m in candidates.items()}
mase = pd.Series({k: r.filter(like="MeanAbsoluteScaledError").iloc[:, 0].mean() for k, r in results.items()})
ax = mase.sort_values(ascending=False).plot.barh(title="Airline MASE across 8 folds (sktime evaluate)")
ax.axvline(1, color="0.4", ls="--");

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
pd.DataFrame({k: r.filter(like="MeanAbsoluteScaledError").iloc[:, 0] for k, r in results.items()}).plot(
    marker="o", ax=ax, title="MASE per fold: rankings are not stable over time")
ax.set_xlabel("fold");

## Takeaways
- sktime is the **glue**: the same API for statsmodels, scikit-learn and LightGBM, and through adapters for neuralforecast, Chronos, TimesFM, …
- Pipelines encode domain knowledge (log, deseasonalize, detrend) explicitly and reversibly.
- Per-fold variability is large. Always look at the distribution, not just the mean.